In [47]:
import pandas as pd
import json

# Read the CSV file
df = pd.read_csv('train.csv')

In [2]:
import pandas as pd
import json

# Read the CSV file
val_df = pd.read_csv('valid.csv')

In [3]:


# Open a new JSONL file to write
with open('translation_data.jsonl', 'w', encoding='utf-8') as f:
    # Iterate through each row in the dataframe
    for _, row in df.iterrows():
        # Create the conversation format for each translation pair
        conversation = [
            {
                "content": f"Translate the following text from Javanese to Indonesian. Return only the translated text: {row['javanese']}", 
                "role": "user"
            },
            {
                "content": f"{row['indonesian']}", 
                "role": "assistant"
            }
        ]
        
        # Write the conversation to the JSONL file
        f.write(json.dumps(conversation, ensure_ascii=False) + '\n')

print("JSONL file has been created successfully!")


JSONL file has been created successfully!


In [2]:
# Open a new JSONL file to write
with open('translation_data_simple_seq2seq_validation.jsonl', 'w', encoding='utf-8') as f:
    # Iterate through each row in the dataframe
    for _, row in val_df.iterrows():
        # Create the text format for each translation pair
        # text = f"Translate the following text from Javanese to Indonesian. Return only the translated text: {row['javanese']}\n{row['indonesian']}"
        
        # Create the data format
        data = {
            "text": row["javanese"],
            "target": row["indonesian"]
        }
        
        # Write to the JSONL file
        f.write(json.dumps(data, ensure_ascii=False) + '\n')

print("Simple JSONL file has been created successfully!")


Simple JSONL file has been created successfully!


In [4]:
# Create lists to store the data
source_texts = []
target_texts = []

# Add Javanese-Indonesian pairs
for _, row in df.iterrows():
    source_texts.append(f"Translate Javanese to Indonesian: {row['javanese']}")
    target_texts.append(row['indonesian'])

# Add Sundanese-Indonesian pairs
for _, row in df.iterrows():
    source_texts.append(f"Translate Sundanese to Indonesian: {row['sundanese']}")
    target_texts.append(row['indonesian'])

# Create a dataframe
translation_df = pd.DataFrame({
    'source_text': source_texts,
    'target_text': target_texts,
})

# Save to CSV
translation_df.to_csv('translation_data_for_llm_ft_snd_jvn.csv', index=False)

print("CSV file has been created successfully!")


CSV file has been created successfully!


Create CSV with just Javanese and Indonesian pairs

In [4]:
# Create lists to store the data
source_texts = []
target_texts = []

# Add Javanese-Indonesian pairs
for _, row in df.iterrows():
    source_texts.append(row['javanese'])
    target_texts.append(row['indonesian'])

# Create a dataframe
translation_df = pd.DataFrame({
    'source_text': source_texts,
    'target_text': target_texts,
})

# Save to CSV
translation_df.to_csv('train_javanese_indonesian.csv', index=False)

print("CSV file has been created successfully!")

CSV file has been created successfully!


### Create extended trainset with more examples - aug_train_jvn_ind.csv

In [5]:
import pandas as pd
import json

# get all data from train_javanese_indonesian.csv
train_jvn_ind = pd.read_csv('train_javanese_indonesian.csv')


In [6]:
# train_jvn_ind = pd.read_csv('train_javanese_indonesian.csv')
# print(train_jvn_ind.to_string(index=False))

In [7]:
from google import genai
from google.genai import types
from pydantic import BaseModel

client = genai.Client(api_key="")

class Entry(BaseModel):
  javanese: str
  indonesian: str

def generate_more_examples(examples):
    response = client.models.generate_content(
        model="gemini-2.0-flash", 
        contents=f"""
        You will help create a more comprehensive training set for a model that translates Javanese to Indonesian.
        Below is a list of Javanese text samples and their Indonesian translations.
        <examples>
        {examples}
        </examples>
        Please generate 50 more rows of Javanese text samples and their Indonesian translations, different from the examples above.
        Each example should range from 1 to 3 sentences.
        The examples should be in a variety of the formality levels, topics, and grammatical structures (not just simple sentence constructions) present in Javanese,
        across the formality levels Ngoko, Ngoko Alus, Krama, and Krama Alus.
        Return the examples in a JSON list of objects with two columns: "javanese" and "indonesian".
        Generate 50 examples.
        """,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=list[Entry],
            safety_settings=[
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                )
            ]
        )
    )
    return response.text

In [8]:
# res = generate_more_examples(train_jvn_ind.to_string(index=False))
# print(res)
# print(json.loads(res))

In [9]:
# call the generate_more_examples function
existing_examples_pd = train_jvn_ind

# for 10 iterations, generate 100 more examples, add them to the existing examples, and save the intermediate results to a csv file
# pass the progressively building examples to the generate_more_examples function
for i in range(20):
    more_examples = generate_more_examples(existing_examples_pd.to_string(index=False))
    # write the more_examples to a csv file
    more_examples_json = json.loads(more_examples)
    # convert the json to a dataframe and csv
    more_examples_pd = pd.DataFrame(more_examples_json)
    more_examples_pd.to_csv(f'2intermediate_aug_train_jvn_ind_{i}.csv', index=False)
    
    existing_examples_pd = pd.concat([existing_examples_pd, more_examples_pd], ignore_index=True)
    existing_examples_pd.to_csv(f'2aug_train_jvn_ind_{i}.csv', index=False)
    print(f'2aug_train_jvn_ind_{i}.csv has been created successfully!')



2aug_train_jvn_ind_0.csv has been created successfully!
2aug_train_jvn_ind_1.csv has been created successfully!
2aug_train_jvn_ind_2.csv has been created successfully!
2aug_train_jvn_ind_3.csv has been created successfully!
2aug_train_jvn_ind_4.csv has been created successfully!
2aug_train_jvn_ind_5.csv has been created successfully!
2aug_train_jvn_ind_6.csv has been created successfully!
2aug_train_jvn_ind_7.csv has been created successfully!
2aug_train_jvn_ind_8.csv has been created successfully!
2aug_train_jvn_ind_9.csv has been created successfully!
2aug_train_jvn_ind_10.csv has been created successfully!
2aug_train_jvn_ind_11.csv has been created successfully!
2aug_train_jvn_ind_12.csv has been created successfully!
2aug_train_jvn_ind_13.csv has been created successfully!
2aug_train_jvn_ind_14.csv has been created successfully!
2aug_train_jvn_ind_15.csv has been created successfully!
2aug_train_jvn_ind_16.csv has been created successfully!
2aug_train_jvn_ind_17.csv has been create

Save augmented dataset to seq2seq jsonl

In [46]:
# read the augmented dataset - ./aug_1_dataset/aug_train_jvn_ind_dataset_1000.csv
aug_train_jvn_ind = pd.read_csv('./aug_1_dataset/aug_train_jvn_ind_dataset_1495.csv')

# create a seq2seq jsonl file
with open('aug_2_train_jvn_ind_seq2seq.jsonl', 'w', encoding='utf-8') as f:
    # write it in the format: {"text": <javanese>, "target": <indonesian>}
    for _, row in aug_train_jvn_ind.iterrows():
        f.write(f'{{"text": "{row["javanese"]}", "target": "{row["indonesian"]}"}}\n')

### Create chain-of-thought extension for the initial dataset

In [85]:
# take 25 rows at a time from the initial dataset, df
# for each row, prompt the model to generate a chain-of-thought for the translation, explaining the steps it took to arrive at the answer
# 
# run the prompt through the model
# write the response to a file
from time import sleep
from google import genai
from google.genai import types
from pydantic import BaseModel

client = genai.Client(api_key="")

class Entry(BaseModel):
  javanese: str
  indonesian: str
  explanation: str

def generate_more_examples(examples):
    response = client.models.generate_content(
        model="gemini-2.0-flash", 
        contents=f"""
        You will help create a more comprehensive training set for a model that translates Javanese to Indonesian,
        incorporating chain-of-thought reasoning.
        Below is a list of Javanese text samples and their Indonesian translations.
        <examples>
        {examples}
        </examples>
        For each example, prompt the model to generate a chain-of-thought for the translation, explaining the steps it took to arrive at the answer,
        with phrase-level explanations.
        Return the explanation in a JSON list of objects with three columns: "javanese", "indonesian", and "explanation", where
        "javanese" is the original Javanese text, "indonesian" is the translated Indonesian text, and "explanation" is the chain-of-thought explanation.
        
        Do this for all of the examples provided.
        """,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=list[Entry],
            safety_settings=[
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                ),
                types.SafetySetting(
                    category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    threshold=types.HarmBlockThreshold.BLOCK_NONE,
                )
            ]
        )
    )
    return response.text

# take 25 rows at a time from the initial dataset, df
# for each row, prompt the model to generate a chain-of-thought for the translation, explaining the steps it took to arrive at the answer
# run the prompt through the model
# write the response to a file
def generate_chain_of_thought(df):
    # take 25 rows at a time from the initial dataset, df
    results = []
    for i in range(0, len(df), 25):
        # for each set of 25 rows, prompt the model to generate a chain-of-thought for the translation, explaining the steps it took to arrive at the answer
        examples = df.iloc[i:i+25]
        # retry 10 times if the model returns an error
        for j in range(10):
            try:
                res = generate_more_examples(examples.to_string(index=False))
                break
            except Exception as e:
                print("error on set of 25 rows", i, "attempt", j, "error", e)
                sleep(5)
                continue

        print(len(examples))
        results.append(res)
        print("completed set of 25 rows", i)
        print("length of results", len(results))
        print("--------------------------------")
    return results


In [86]:
output = generate_chain_of_thought(df)

25
completed set of 25 rows 0
length of results 1
--------------------------------
error on set of 25 rows 25 attempt 0 error 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
error on set of 25 rows 25 attempt 1 error 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
error on set of 25 rows 25 attempt 2 error 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
25
completed set of 25 rows 25
length of results 2
--------------------------------
25
completed set of 25 rows 50
length of results 3
--------------------------------
error on set of 25 rows 75 attempt 0 error 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}
error on set of 25 rows 75 attempt 1 error 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The s

In [87]:
json.loads(output[0])

[{'javanese': 'Nikmatono cicilan 0% sampek 12 sasi dinggo pesen tiket kapal air asia nganggo kertu kredit bni!',
  'indonesian': 'Nikmati cicilan 0% hingga 12 bulan untuk pemesanan tiket pesawat air asia dengan kartu kredit bni!',
  'explanation': 'The Javanese sentence discusses enjoying 0% installments for up to 12 months when ordering Air Asia tickets using a BNI credit card. "Nikmatono" translates to "Nikmati" (Enjoy). "cicilan 0%" is the same as in Indonesian. "sampek 12 sasi" translates to "hingga 12 bulan" (up to 12 months). "dinggo pesen tiket kapal air asia" translates to "untuk pemesanan tiket pesawat Air Asia" (for ordering Air Asia airplane tickets). "nganggo kertu kredit bni" translates to "dengan kartu kredit BNI" (with a BNI credit card).'},
 {'javanese': 'Roti-roti sing disajekne nggarai aku nostalgianan. Kabeh model roti jaman biyen, saka tampilane utawa rasa. Rotine enak lan regane uga mirah.',
  'indonesian': 'Kue-kue yang disajikan bikin saya bernostalgia. Semuanya 

In [88]:
# combine the results into a single json list
combined_results = []
for result in output:
    combined_results.extend(json.loads(result))


# save the combined results to a jsonl file
with open('chain_of_thought_results.jsonl', 'w', encoding='utf-8') as f:
    for result in combined_results:
        f.write(json.dumps(result, ensure_ascii=False) + '\n')

# remove duplicates by converting each dict to a tuple of items, then back to dict
unique_dicts = {}
for item in combined_results:
    # Convert dict to a hashable representation (tuple of sorted items)
    dict_key = tuple(sorted(item.items()))
    unique_dicts[dict_key] = item

# Get the unique dictionaries
combined_results = list(unique_dicts.values())

# save the combined results to a jsonl file
with open('chain_of_thought_results_no_duplicates.jsonl', 'w', encoding='utf-8') as f:
    for result in combined_results:
        f.write(json.dumps(result, ensure_ascii=False) + '\n')


In [90]:
# iterate through the chain of thought results and add them to the initial dataset,
# making a jsonl with two fields: "text" and "target"
# for examples from initial dataset, strcuture it as:
# {"text": Translate from Javanese to Indonesian: <javanese>, "target": <indonesian>}
# for chain of thought results, structure it as:
# {"text": Translate from Javanese to Indonesian, and then explain the steps: <javanese>, "target": <indonesian> | <explanation>}

# read the initial dataset
initial_dataset = pd.read_csv('train_javanese_indonesian.csv')

# read the chain of thought results
chain_of_thought_results = pd.read_json('chain_of_thought_results_no_duplicates.jsonl', lines=True)

with open('combined_dataset.jsonl', 'w', encoding='utf-8') as f:
    for index, initial_row in initial_dataset.iterrows():
        f.write(json.dumps({"text": f"Translate from Javanese to Indonesian: {initial_row['javanese']}", "target": initial_row['indonesian']}, ensure_ascii=False) + '\n')
        chain_of_thought_row = chain_of_thought_results.iloc[index]
        f.write(json.dumps({"text": f"Translate from Javanese to Indonesian, and explain how it was translated: {chain_of_thought_row['javanese']}", "target": f"{chain_of_thought_row['indonesian']} | {chain_of_thought_row['explanation']}"}, ensure_ascii=False) + '\n')

# save the combined dataset to a csv
combined_dataset = pd.read_json('combined_dataset.jsonl', lines=True)
combined_dataset.to_csv('combined_dataset.csv', index=False)


In [91]:
def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['text'])):
        # Format for Gemma chat template
        text = [
                {
                    "role": "user",
                    "content": [{"type": "text", "text": example['text'][i]}]
                },
                {
                    "role": "model",
                    "content": [{"type": "text", "text": example['target'][i]}]
                }
            ]
        output_texts.append(text)
    return output_texts

In [92]:
formatted_prompts = formatting_prompts_func(combined_dataset)

In [94]:
formatted_prompts[1]

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Translate from Javanese to Indonesian, and explain how it was translated: Nikmatono cicilan 0% sampek 12 sasi dinggo pesen tiket kapal air asia nganggo kertu kredit bni!'}]},
 {'role': 'model',
  'content': [{'type': 'text',
    'text': 'Nikmati cicilan 0% hingga 12 bulan untuk pemesanan tiket pesawat air asia dengan kartu kredit bni! | The Javanese sentence discusses enjoying 0% installments for up to 12 months when ordering Air Asia tickets using a BNI credit card. "Nikmatono" translates to "Nikmati" (Enjoy). "cicilan 0%" is the same as in Indonesian. "sampek 12 sasi" translates to "hingga 12 bulan" (up to 12 months). "dinggo pesen tiket kapal air asia" translates to "untuk pemesanan tiket pesawat Air Asia" (for ordering Air Asia airplane tickets). "nganggo kertu kredit bni" translates to "dengan kartu kredit BNI" (with a BNI credit card).'}]}]

### CoT part 2
Use the chain of thought from previous step to create new dataset format

In [98]:
# new dataset format:
# {"text": Here is a Javanese sentence: <javanese> and its Indonesian translation: <indonesian>. Now, explain how it was translated: <explanation>}
# {"text": Translate from Javanese to Indonesian: <javanese>, "target": <indonesian>}

# read the initial dataset
initial_dataset = pd.read_csv('train_javanese_indonesian.csv')

# read the chain of thought results
chain_of_thought_results = pd.read_json('chain_of_thought_results_no_duplicates.jsonl', lines=True)

# create a new dataset
new_dataset = []

# for each row in the initial dataset, create a new row in the new dataset
for index, row in initial_dataset.iterrows():
    # get the javanese and indonesian from the row
    javanese = row['javanese']
    indonesian = row['indonesian']
    explanation = chain_of_thought_results.iloc[index]['explanation']
    # create a new row in the new dataset
    new_dataset.append({'text': f"Here is a Javanese sentence: {javanese} and its Indonesian translation: {indonesian}. Now, explain how it was translated.", 'target': f"{explanation}"})
    new_dataset.append({'text': f"Translate from Javanese to Indonesian: {javanese}", 'target': f"{indonesian}"})

# save the new dataset to a csv
new_dataset_df = pd.DataFrame(new_dataset)
new_dataset_df.to_csv('combined_dataset_v2.csv', index=False)
# save the new dataset to a jsonl
with open('combined_dataset_v2.jsonl', 'w', encoding='utf-8') as f:
    for index, row in new_dataset_df.iterrows():
        f.write(json.dumps({"text": row['text'], "target": row['target']}, ensure_ascii=False) + '\n')


